# nb23 — Model Selection: XGBoost vs CatBoost vs Random Forest

All three models are competitive in nb21. This notebook makes the final call using:
1. **Paired t-test** on 5-fold CV AUCs — are differences statistically real or noise?
2. **Lift curves** — which model finds more hitmakers at each screening threshold?
3. **Disagreement analysis** — where do models split, and who is right?
4. **Summary recommendation**

Models use their **nb21 final params** (no re-tuning).  
CatBoost uses the **dedicated nb18 Mid params** (n=12, 100-trial Optuna), not the pipeline version.

| Model | Test AUC | Gap | Recall | F1 | Leaves |
|---|---|---|---|---|---|
| XGBoost | 0.774 | −0.003 | 0.727 | 0.686 | 468 |
| Random Forest | 0.767 | +0.005 | 0.758 | 0.680 | 1590 |
| CatBoost (pipeline) | 0.753 | +0.021 | 0.697 | 0.639 | 1430 |
| **CatBoost (dedicated)** | **0.770** | **+0.021** | **0.773** | **0.680** | **400** |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
N_SPLITS     = 5

In [ ]:
# Data loading — identical to nb22.
df = pd.read_csv('df_artists_final.csv', index_col=0).reset_index()
X_raw = df.drop(columns=['top_20_hitmaker'])
y     = df['top_20_hitmaker']

ALL_GENRE_COLS = [c for c in X_raw.columns if c.startswith('artist_genre_')]

def add_genre_other(df_imp, keep_genres):
    """Sum low-signal genre dummies into artist_genre_other."""
    keep_full  = [f'artist_genre_{g}' if not g.startswith('artist_genre_') else g
                  for g in keep_genres]
    low_signal = [c for c in ALL_GENRE_COLS if c not in keep_full]
    out = df_imp.copy()
    out['artist_genre_other'] = (df_imp[low_signal].sum(axis=1) > 0).astype(int)
    return out

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_raw),
                            columns=X_train_raw.columns, index=X_train_raw.index)
X_test_imp  = pd.DataFrame(imputer.transform(X_test_raw),
                            columns=X_test_raw.columns,  index=X_test_raw.index)

print(f'Train: {X_train_imp.shape}  Test: {X_test_imp.shape}')
print(f'Class balance (train): {y_train.value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# ── Feature sets (nb21 final selections) ──────────────────────────────────────
XGB_FEATURES = [
    '#_of_charting_songs_through_first_top_20_hit',
    '#_of_genres_artist',
    'artist_genre_Pop',
    'artist_genre_Hip Hop/Rap',
    'top_20_hit_song_#_wks_on_chart_any_position',
    'artist_genre_R&B/Soul/Funk',
    'artist_genre_unknown',
]
RF_FEATURES = [
    '#_of_charting_songs_through_first_top_20_hit',
    '#_of_genres_artist',
    'artist_genre_Hip Hop/Rap',
    'artist_genre_Pop',
    'artist_genre_unknown',
    'top_20_hit_song_#_wks_on_chart_any_position',
    'artist_genre_R&B/Soul/Funk',
    'years_through_first_top_20_hit',
    'artist_genre_other',          # requires add_genre_other()
]
CAT_FEATURES = [
    '#_of_charting_songs_through_first_top_20_hit',
    '#_of_genres_artist',
    'betweenness_centrality_top20_rolling5',
    'artist_genre_Pop',
    'harmonic_closeness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
    'years_through_first_top_20_hit',
    'artist_genre_Hip Hop/Rap',
    'top_20_hit_song_#_wks_on_chart_any_position',
    'artist_genre_other',          # requires add_genre_other()
    'artist_genre_R&B/Soul/Funk',
    'artist_genre_Rock',
]

# ── Model params ──────────────────────────────────────────────────────────────
XGB_PARAMS = {
    'n_estimators': 239, 'learning_rate': 0.0238, 'max_depth': 5,
    'min_child_weight': 12, 'gamma': 4.5, 'subsample': 0.62,
    'colsample_bytree': 0.36, 'reg_alpha': 0.36, 'reg_lambda': 0.0,
    'random_state': RANDOM_STATE, 'eval_metric': 'logloss', 'verbosity': 0,
}
RF_PARAMS = {
    'n_estimators': 409, 'max_depth': 2, 'min_samples_leaf': 17,
    'max_features': 'sqrt', 'class_weight': 'balanced',
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}
CAT_PARAMS = {
    'n_estimators': 50, 'learning_rate': 0.06180643097470682,
    'depth': 3, 'l2_leaf_reg': 4.970472180919757,
    'random_strength': 3.657777929321733, 'min_data_in_leaf': 9,
    'border_count': 178, 'random_state': RANDOM_STATE, 'verbose': 0,
}

# ── Build datasets with artist_genre_other ───────────────────────────────────
CAT_KEEP = ['artist_genre_Pop', 'artist_genre_Hip Hop/Rap',
            'artist_genre_R&B/Soul/Funk', 'artist_genre_Rock']
RF_KEEP  = ['artist_genre_Pop', 'artist_genre_Hip Hop/Rap',
            'artist_genre_R&B/Soul/Funk', 'artist_genre_unknown']

X_train_cat = add_genre_other(X_train_imp, CAT_KEEP)
X_test_cat  = add_genre_other(X_test_imp,  CAT_KEEP)
X_train_rf  = add_genre_other(X_train_imp, RF_KEEP)
X_test_rf   = add_genre_other(X_test_imp,  RF_KEEP)

print('Feature counts — XGB:', len(XGB_FEATURES),
      ' RF:', len(RF_FEATURES),
      ' CatBoost:', len(CAT_FEATURES))

In [ ]:
# ── 5-fold CV storing per-fold AUCs (needed for paired t-test) ───────────────
def cv_fold_aucs(X_full, y_full, model_name, features, genre_keep=None):
    """Returns list of (train_auc, val_auc) per fold."""
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_results = []
    for tr_idx, val_idx in skf.split(X_full, y_full):
        X_tr_raw = X_full.iloc[tr_idx]
        X_vl_raw = X_full.iloc[val_idx]
        y_tr     = y_full.iloc[tr_idx]
        y_vl     = y_full.iloc[val_idx]

        imp = SimpleImputer(strategy='median')
        X_tr = pd.DataFrame(imp.fit_transform(X_tr_raw), columns=X_tr_raw.columns)
        X_vl = pd.DataFrame(imp.transform(X_vl_raw),     columns=X_vl_raw.columns)

        if genre_keep is not None:
            X_tr = add_genre_other(X_tr, genre_keep)
            X_vl = add_genre_other(X_vl, genre_keep)

        if model_name == 'xgb':
            m = XGBClassifier(**XGB_PARAMS)
        elif model_name == 'rf':
            m = RandomForestClassifier(**RF_PARAMS)
        else:
            m = CatBoostClassifier(**CAT_PARAMS)

        m.fit(X_tr[features], y_tr)
        tr_auc  = roc_auc_score(y_tr, m.predict_proba(X_tr[features])[:, 1])
        val_auc = roc_auc_score(y_vl, m.predict_proba(X_vl[features])[:, 1])
        fold_results.append((tr_auc, val_auc))
    return fold_results

print('Running 5-fold CV for all three models...')
xgb_folds = cv_fold_aucs(X_raw, y, 'xgb', XGB_FEATURES)
print('  XGBoost done')
rf_folds  = cv_fold_aucs(X_raw, y, 'rf',  RF_FEATURES,  genre_keep=RF_KEEP)
print('  Random Forest done')
cat_folds = cv_fold_aucs(X_raw, y, 'cat', CAT_FEATURES, genre_keep=CAT_KEEP)
print('  CatBoost done')

xgb_val = [v for _, v in xgb_folds]
rf_val  = [v for _, v in rf_folds]
cat_val = [v for _, v in cat_folds]

print(f'\nPer-fold Val AUC:')
print(f'{"Fold":<6} {"XGBoost":>10} {"RF":>10} {"CatBoost":>10}')
for i, (x, r, c) in enumerate(zip(xgb_val, rf_val, cat_val), 1):
    print(f'{i:<6} {x:>10.4f} {r:>10.4f} {c:>10.4f}')
print(f'{"Mean":<6} {np.mean(xgb_val):>10.4f} {np.mean(rf_val):>10.4f} {np.mean(cat_val):>10.4f}')
print(f'{"Std":<6} {np.std(xgb_val):>10.4f} {np.std(rf_val):>10.4f} {np.std(cat_val):>10.4f}')

In [ ]:
# ── Paired t-test: is the AUC difference statistically significant? ───────────
# H0: mean AUC difference = 0.  df = N_SPLITS - 1 = 4.
# With 5 folds, power is low — treat p < 0.10 as suggestive, p < 0.05 as significant.

pairs = [
    ('XGBoost vs RF',       xgb_val, rf_val),
    ('XGBoost vs CatBoost', xgb_val, cat_val),
    ('RF vs CatBoost',      rf_val,  cat_val),
]

print(f'Paired t-test (5-fold CV, df=4)')
print(f'{"Comparison":<25} {"Mean diff":>10} {"t-stat":>8} {"p-value":>10} {"Verdict"}')
print('-' * 70)
for label, a, b in pairs:
    diff   = np.array(a) - np.array(b)
    t, p   = stats.ttest_rel(a, b)
    mean_d = np.mean(diff)
    if p < 0.05:
        verdict = '** significant'
    elif p < 0.10:
        verdict = '*  suggestive'
    else:
        verdict = '   not significant'
    print(f'{label:<25} {mean_d:>+10.4f} {t:>8.3f} {p:>10.4f}   {verdict}')

print()
print('Note: with only 5 folds, the t-test has limited power (df=4).')
print('A non-significant result means the difference could be noise — not that models are equal.')
print('A significant result is a stronger signal given the small sample.')

# Visualize fold AUCs
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(1, N_SPLITS + 1)
ax.plot(x, xgb_val, 'o-', color='#2166ac', label=f'XGBoost  (mean={np.mean(xgb_val):.4f})', lw=2)
ax.plot(x, rf_val,  's-', color='#4dac26', label=f'Random Forest (mean={np.mean(rf_val):.4f})', lw=2)
ax.plot(x, cat_val, '^-', color='#d73027', label=f'CatBoost (mean={np.mean(cat_val):.4f})', lw=2)
ax.set_xlabel('Fold', fontsize=11)
ax.set_ylabel('Validation AUC', fontsize=11)
ax.set_title('Per-fold validation AUC — XGBoost vs RF vs CatBoost',
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_xticks(x)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Fit final models on full training set for lift + disagreement analysis ────
xgb = XGBClassifier(**XGB_PARAMS)
xgb.fit(X_train_imp[XGB_FEATURES], y_train)

rf  = RandomForestClassifier(**RF_PARAMS)
rf.fit(X_train_rf[RF_FEATURES], y_train)

cat = CatBoostClassifier(**CAT_PARAMS)
cat.fit(X_train_cat[CAT_FEATURES], y_train)

xgb_prob = xgb.predict_proba(X_test_imp[XGB_FEATURES])[:, 1]
rf_prob  = rf.predict_proba(X_test_rf[RF_FEATURES])[:, 1]
cat_prob = cat.predict_proba(X_test_cat[CAT_FEATURES])[:, 1]

xgb_pred = (xgb_prob >= 0.44).astype(int)
rf_pred  = (rf_prob  >= 0.49).astype(int)
cat_pred = (cat_prob >= 0.40).astype(int)

print('Test set metrics (OOF-tuned thresholds from nb21):')
print(f'{"":<14} {"AUC":>7} {"F1":>7} {"Precision":>10} {"Recall":>8}')
for name, prob, pred in [
    ('XGBoost',      xgb_prob, xgb_pred),
    ('Random Forest',rf_prob,  rf_pred),
    ('CatBoost',     cat_prob, cat_pred),
]:
    print(f'{name:<14} {roc_auc_score(y_test, prob):>7.4f} '
          f'{f1_score(y_test, pred):>7.4f} '
          f'{precision_score(y_test, pred):>10.4f} '
          f'{recall_score(y_test, pred):>8.4f}')

In [ ]:
# ── Lift curves — all three models ───────────────────────────────────────────
def compute_lift(y_true, y_score):
    order    = np.argsort(y_score)[::-1]
    y_sorted = np.array(y_true)[order]
    base     = np.mean(y_true)
    lifts, fracs = [], []
    for k in range(1, len(y_true) + 1):
        lifts.append(y_sorted[:k].mean() / base)
        fracs.append(k / len(y_true))
    return np.array(fracs), np.array(lifts)

fig, ax = plt.subplots(figsize=(9, 6))
for prob, label, color, ls in [
    (xgb_prob, f'XGBoost  (AUC={roc_auc_score(y_test, xgb_prob):.3f})', '#2166ac', '-'),
    (rf_prob,  f'Random Forest (AUC={roc_auc_score(y_test, rf_prob):.3f})',  '#4dac26', '--'),
    (cat_prob, f'CatBoost (AUC={roc_auc_score(y_test, cat_prob):.3f})', '#d73027', '-.'),
]:
    fracs, lifts = compute_lift(y_test.values, prob)
    ax.plot(fracs * 100, lifts, label=label, color=color, lw=2, ls=ls)

ax.axhline(1.0, color='gray', ls=':', lw=1, label='Random baseline')
for pct in [10, 20, 30]:
    ax.axvline(pct, color='black', ls=':', lw=0.8, alpha=0.4)
    ax.text(pct + 0.5, ax.get_ylim()[0] + 0.05, f'{pct}%', fontsize=8, color='gray')

ax.set_xlabel('% of artists screened (ranked by score)', fontsize=11)
ax.set_ylabel('Lift over random', fontsize=11)
ax.set_title('Lift Curve — Who finds more hitmakers at each screening cutoff?',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\n{"Screening %":<14} {"XGBoost":>10} {"RF":>10} {"CatBoost":>10}')
for pct in [10, 20, 30, 50]:
    idx = int(pct / 100 * len(y_test)) - 1
    fx, lx = compute_lift(y_test.values, xgb_prob)
    fr, lr = compute_lift(y_test.values, rf_prob)
    fc, lc = compute_lift(y_test.values, cat_prob)
    print(f'Top {pct:2d}%        {lx[idx]:>10.2f}x {lr[idx]:>10.2f}x {lc[idx]:>10.2f}x')

In [ ]:
# ── Disagreement analysis ─────────────────────────────────────────────────────
# Build a per-artist prediction table
pred_df = pd.DataFrame({
    'true':     y_test.values,
    'xgb_prob': xgb_prob,
    'rf_prob':  rf_prob,
    'cat_prob': cat_prob,
    'xgb_pred': xgb_pred,
    'rf_pred':  rf_pred,
    'cat_pred': cat_pred,
}, index=y_test.index)

# Agreement categories
pred_df['n_hitmaker_votes'] = pred_df[['xgb_pred', 'rf_pred', 'cat_pred']].sum(axis=1)
pred_df['unanimous']        = pred_df['n_hitmaker_votes'].isin([0, 3])
pred_df['split']            = ~pred_df['unanimous']
pred_df['avg_prob']         = pred_df[['xgb_prob', 'rf_prob', 'cat_prob']].mean(axis=1)

n_unanimous = pred_df['unanimous'].sum()
n_split     = pred_df['split'].sum()
print(f'Unanimous predictions:  {n_unanimous}/{len(pred_df)} ({n_unanimous/len(pred_df):.0%})')
print(f'Split predictions:      {n_split}/{len(pred_df)} ({n_split/len(pred_df):.0%})')
print()

# Accuracy on unanimous vs split
for label, mask in [('Unanimous', pred_df['unanimous']),
                    ('Split (2-1)',  pred_df['split'])]:
    sub = pred_df[mask]
    # majority vote prediction for split
    maj_pred = (sub['n_hitmaker_votes'] >= 2).astype(int)
    acc  = (maj_pred == sub['true']).mean()
    n_tp = ((maj_pred == 1) & (sub['true'] == 1)).sum()
    n_fp = ((maj_pred == 1) & (sub['true'] == 0)).sum()
    n_tn = ((maj_pred == 0) & (sub['true'] == 0)).sum()
    n_fn = ((maj_pred == 0) & (sub['true'] == 1)).sum()
    print(f'{label} (n={mask.sum()}):')
    print(f'  Majority-vote accuracy: {acc:.3f}   TP={n_tp} FP={n_fp} TN={n_tn} FN={n_fn}')

# Which model is right more often on split cases?
split = pred_df[pred_df['split']].copy()
print(f'\nOn split cases (n={len(split)}), individual model accuracy:')
for name, col in [('XGBoost', 'xgb_pred'), ('RF', 'rf_pred'), ('CatBoost', 'cat_pred')]:
    acc = (split[col] == split['true']).mean()
    print(f'  {name:<14}: {acc:.3f}')

# Visualize: avg probability distribution for unanimous vs split cases
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, mask, title, color in [
    (axes[0], pred_df['unanimous'], 'Unanimous predictions', '#2166ac'),
    (axes[1], pred_df['split'],     'Split predictions (2 vs 1)', '#d73027'),
]:
    sub = pred_df[mask]
    ax.scatter(sub['avg_prob'],
               np.random.uniform(-0.1, 0.1, len(sub)),
               c=sub['true'], cmap='RdBu_r', alpha=0.5, s=30, vmin=0, vmax=1)
    ax.axvline(0.5, color='gray', ls='--', lw=1)
    ax.set_xlabel('Average probability across 3 models', fontsize=10)
    ax.set_yticks([])
    ax.set_title(f'{title}\n(n={mask.sum()}, blue=one-hit wonder, red=hitmaker)',
                 fontsize=10, fontweight='bold')
    ax.spines[['top', 'right', 'left']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
from sklearn.metrics import log_loss, brier_score_loss

summary = pd.DataFrame([
    {'Model': 'XGBoost',
     'CV AUC': f'{np.mean(xgb_val):.4f} ± {np.std(xgb_val):.4f}',
     'Test AUC': f'{roc_auc_score(y_test, xgb_prob):.4f}',
     'F1': f'{f1_score(y_test, xgb_pred):.3f}',
     'Precision': f'{precision_score(y_test, xgb_pred):.3f}',
     'Recall': f'{recall_score(y_test, xgb_pred):.3f}',
     'Logloss': f'{log_loss(y_test, xgb_prob):.4f}',
     'N Features': 7, 'Trees': 239, 'Depth': 5, 'Leaves': 468},
    {'Model': 'Random Forest',
     'CV AUC': f'{np.mean(rf_val):.4f} ± {np.std(rf_val):.4f}',
     'Test AUC': f'{roc_auc_score(y_test, rf_prob):.4f}',
     'F1': f'{f1_score(y_test, rf_pred):.3f}',
     'Precision': f'{precision_score(y_test, rf_pred):.3f}',
     'Recall': f'{recall_score(y_test, rf_pred):.3f}',
     'Logloss': f'{log_loss(y_test, rf_prob):.4f}',
     'N Features': 9, 'Trees': 409, 'Depth': 2, 'Leaves': 1590},
    {'Model': 'CatBoost (dedicated)',
     'CV AUC': f'{np.mean(cat_val):.4f} ± {np.std(cat_val):.4f}',
     'Test AUC': f'{roc_auc_score(y_test, cat_prob):.4f}',
     'F1': f'{f1_score(y_test, cat_pred):.3f}',
     'Precision': f'{precision_score(y_test, cat_pred):.3f}',
     'Recall': f'{recall_score(y_test, cat_pred):.3f}',
     'Logloss': f'{log_loss(y_test, cat_prob):.4f}',
     'N Features': 12, 'Trees': 50, 'Depth': 3, 'Leaves': 400},
]).set_index('Model')

print('=== Final Comparison ===')
print(summary.to_string())
print()
print('=== Paired t-test summary ===')
for label, a, b in pairs:
    t, p = stats.ttest_rel(a, b)
    print(f'  {label:<25}  mean diff={np.mean(np.array(a)-np.array(b)):+.4f}  p={p:.4f}')

## Reading the results

**If XGBoost vs RF t-test is NOT significant (p > 0.10):**  
The two models are statistically tied on CV AUC. Decision falls to other criteria:
- Prefer **RF** if the goal is recall (finding hitmakers) — RF had best recall in nb21 (0.758)
- Prefer **XGBoost** if the goal is precision or explainability — fewer features, negative gap
- Prefer **CatBoost** if the goal is a symmetric-tree explanation (50 trees, depth=3, 400 leaves)

**If XGBoost vs RF t-test IS significant (p < 0.05):**  
XGBoost is reliably better and RF is ruled out.

**On the disagreement analysis:**  
High accuracy on unanimous cases confirms the models agree where it's easy.  
Low accuracy on split cases is expected — those are the genuinely ambiguous artists.  
If one model is consistently right on split cases, that's evidence it captures something the others miss.